# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Paper finding 1: Anatomy of Growing Content (Finding #1)

Finding (paraphrased): pages with rising impressions are about 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than pages with falling impressions (74,187 rising vs 45,272 falling).

Methodology questions I would ask:
- The paper itself reports that content age and word count are negatively correlated (r = -0.52). Since the growing pages are also younger, how much of the word-count gap remains once age is held fixed (for example, comparing within the same age band)?
- Trend direction comes from a 30-day vs previous-30-day impression change. Could very young pages be labelled "up" partly because they start from a tiny base? A check excluding the "New" group or using a minimum-impression floor would help.
- The comparison reports averages only. An effect size with an interval, or medians, would show how much the two groups overlap.

Why it matters: the paper rightly calls this an observational comparison. These checks would show how much of the gap points to depth itself rather than to age.

### Paper finding 2: The Freshness Multiplier (Finding #4)

Finding (paraphrased): 365+ day pages that were refreshed within the last 30 days show about 3.2x higher health score (10.7 to 34.5) and about 57x more impressions (71 to 4,039).

Methodology questions I would ask:
- Are the "before" and "after" numbers the same pages over time, or two different groups of pages (refreshed vs not refreshed)? If different groups, how were pages chosen for a refresh? Pages picked for a refresh may already have had more potential, which would inflate the effect.
- Health score includes impressions (30 of 100 points), so the health gain and the impression gain are partly the same signal. Would a raw-metric comparison (clicks, position) tell a separate story?
- The analysis uses active pages only (impressions and sessions above 0). Old pages that died completely are excluded, so the old stale group may look better or worse than reality. The paper flags this survivor bias for one cell; does it also affect this comparison? What are the group sizes?

Why it matters: the paper keeps the careful reading "older pages can recover when updated well". A matched before/after comparison with a control group of untouched old pages would show how much of the 3.2x and 57x is the refresh itself.

In [1]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/mahmarani/flyrank-ml-internship.git
    %cd flyrank-ml-internship
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.base import clone

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df.impressions_90d > 0].drop_duplicates("content_id").reset_index(drop=True)
df["y"] = (df.trend_direction.str.lower() == "down").astype(int)

num = ["search_volume","competition","cpc","word_count","char_count","impressions_90d","clicks_90d",
       "sessions_90d","engaged_sessions_90d","ai_sessions_90d","days_with_impressions","days_with_sessions",
       "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
cat = ["competition_level","content_type","main_intent","freshness_tier","position_tier","impression_tier"]
X_cols = num + cat

def precision_at_k(y, score, k):
    return float(np.asarray(y)[np.argsort(-np.asarray(score))[:k]].mean())

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
lr = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])
print("Setup done:", df.shape, "| declining rate:", round(df.y.mean(), 3))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 136 (delta 44), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.87 MiB | 13.65 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/flyrank-ml-internship
Setup done: (30000, 45) | declining rate: 0.542


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/mahmarani/flyrank-ml-internship.git
    %cd flyrank-ml-internship
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.base import clone

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df.impressions_90d > 0].drop_duplicates("content_id").reset_index(drop=True)
df["y"] = (df.trend_direction.str.lower() == "down").astype(int)
num = ["search_volume","competition","cpc","word_count","char_count","impressions_90d","clicks_90d",
       "sessions_90d","engaged_sessions_90d","ai_sessions_90d","days_with_impressions","days_with_sessions",
       "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
cat = ["competition_level","content_type","main_intent","freshness_tier","position_tier","impression_tier"]
X_cols = num + cat

def precision_at_k(y, score, k):
    return float(np.asarray(y)[np.argsort(-np.asarray(score))[:k]].mean())

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
lr = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])

rows = []
for seed in range(10):
    splits = {
        "BEFORE: random row split": train_test_split(np.arange(len(df)), test_size=0.25, random_state=seed, stratify=df.y),
        "AFTER: grouped by client": next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                                         .split(df, df.y, groups=df.client_id)),
    }
    for name, (tr, te) in splits.items():
        a, b = df.iloc[tr], df.iloc[te]
        m = clone(lr).fit(a[X_cols], a.y)
        s = m.predict_proba(b[X_cols])[:, 1]
        rows.append({"split": name, "P@50": precision_at_k(b.y, s, 50),
                     "P@100": precision_at_k(b.y, s, 100), "AP": average_precision_score(b.y, s)})
ba = pd.DataFrame(rows).groupby("split")[["P@50", "P@100", "AP"]].agg(["mean", "std"]).round(3)
ba


P@50         P@100            AP       
                           mean    std   mean    std   mean    std
split                                                             
AFTER: grouped by client  0.770  0.089  0.745  0.089  0.647  0.072
BEFORE: random row split  0.874  0.038  0.836  0.026  0.702  0.009

Before/after: same Logistic Regression, same features, 10 seeds. BEFORE uses a random row split, so pages from the same client can appear in both train and test. AFTER groups by client_id, so test clients are never seen in training.

Measured: P@50 falls from 0.874 to 0.770, P@100 from 0.836 to 0.745, and average precision from 0.702 to 0.647. The variation between seeds also grows (P@50 std 0.038 to 0.089). Directional reading: the random split overstates how well the model works on a new client, because it can learn client-specific patterns. The grouped number is the honest one for the question "does this rank well on a client it has never seen?". Even the grouped result is based on only 32 clients, so it stays directional. Decision-support only.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
forbidden = ["trend_pct", "trend_direction", "clicks_last_30d", "sessions_last_30d", "impressions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
print("Forbidden columns in features:", [c for c in X_cols if c in forbidden])   # must be []

# single-feature check: a feature that alone predicts the label very well is suspicious
sf = {}
for c in num:
    v = df[c].fillna(df[c].median())
    sf[c] = max(average_precision_score(df.y, v), average_precision_score(df.y, -v))
print("\nBase rate:", round(df.y.mean(), 3))
print(pd.Series(sf).sort_values(ascending=False).head(8).round(3))

# window overlap: 90d columns contain the last-30d window the label is built from
print("\ncorr(impressions_90d, impressions_last_30d):", round(df.impressions_90d.corr(df.impressions_last_30d), 3))
print("corr(days_with_impressions, y):", round(df.days_with_impressions.corr(df.y), 3))

# client concentration
print("\nTop-5 client row shares:")
print(df.client_id.value_counts(normalize=True).head(5).round(3))

Forbidden columns in features: []

Base rate: 0.542
content_age_days          0.608
search_volume             0.587
days_with_impressions     0.577
word_count                0.575
impressions_90d           0.571
days_since_last_update    0.570
char_count                0.570
competition               0.546
dtype: float64

corr(impressions_90d, impressions_last_30d): 0.918
corr(days_with_impressions, y): 0.19

Top-5 client row shares:
client_id
client_19581e27de    0.234
client_6208ef0f77    0.123
client_4e07408562    0.076
client_3fdba35f04    0.076
client_f369cb89fc    0.060
Name: proportion, dtype: float64


Leakage audit on the final feature set.

1. Direct leakage: no label-derived or future-window columns (trend_pct, trend_direction, *_last_30d, *_prev_30d) are in the features. Checked in code; the list is empty.

2. Single-feature check: the strongest single feature, content_age_days, reaches average precision 0.608 against a base rate of 0.542, and the next ones (search_volume 0.587, days_with_impressions 0.577) are close behind. No single feature separates the classes on its own, which is what we would expect if nothing leaks directly.

3. Window overlap: impressions_90d correlates 0.918 with impressions_last_30d. The 90-day columns contain the last 30 days that define the label, so some indirect overlap remains. I treat this as a limitation, not as proof of leakage. A cleaner design would use only the days before the last 30.

4. Client concentration: the largest client holds 23.4% of rows and the top 5 clients hold about 57%. The grouped split keeps clients apart, but a few clients can still drive the result, so the grouped estimate depends on which clients land in the test set (std of P@50 is 0.089 across seeds).

Verdict: no direct label leakage found; partial window overlap and client concentration are disclosed limitations. Results are directional, decision-support only.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim rewrite.

My boldest sentence (Week 5): "Logistic Regression is the better model and predicts which pages are declining."

Rewrite: "On held-out clients (grouped split, 10 seeds), Logistic Regression ranked pages with a declining 30-day trend somewhat better than the simple rule (mean P@50 0.77 vs 0.67, directional; the gap is about one standard deviation). Under a random row split the same model looked better (P@50 0.87), so that number overstates performance on a new client. The result is decision-support for ordering reviews, not a prediction of Google behavior, and it rests on 32 clients with partial overlap between the 90-day features and the 30-day label."

Failure examples (top-50, one grouped split): 15 of 50 are not declining. Of the 8 shown, 4 are clearly growing (trend +71% to +214%) and 2 have no trend value (very few impressions). The model cannot see recent direction because trend and last-30-day columns are excluded to avoid leakage, so it can rank a recovering page as at-risk. Several others have CTR near 0.00 with modest impressions. Because the label is a single 30-day change, some of these pages may also reflect noise. Next step: add a time-aware split and features built only from days before the last 30.

In [4]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(df, df.y, groups=df.client_id))
a, b = df.iloc[tr], df.iloc[te].copy()
m = clone(lr).fit(a[X_cols], a.y)
b["score"] = m.predict_proba(b[X_cols])[:, 1]
top = b.sort_values("score", ascending=False).head(50)
print("False positives in top-50:", int((top.y == 0).sum()))
top[top.y == 0].head(8)[["content_id", "avg_position", "impressions_90d", "ctr", "trend_pct", "score"]]

False positives in top-50: 15


,content_id,avg_position,impressions_90d,ctr,trend_pct,score
10175,content_374e795aab68,31.0,235,0.85,0.0,0.955544
26614,content_7be5f150dc65,5.9,290,0.00,213.9,0.940758
11668,content_e02b544b7a9a,6.2,5,0.00,NaN,0.907907
3467,content_e331f45208a0,9.2,6,0.00,NaN,0.905258
23512,content_c8a3b125787c,17.3,124,0.00,11.1,0.902759
8016,content_c94a53e3bfb8,8.1,2164,0.23,71.0,0.902666
11089,content_a058e966b9a6,3.2,79,1.27,181.3,0.901148
28293,content_619acf4bbcc3,7.2,181,0.00,147.7,0.896992


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.